# LoreForge — Training Pipeline
**Author:** Spencer Buehlman  
**Course:** Generative AI — Northwestern MSAI

Run this notebook on Quest to execute the full training pipeline:
data download → tokenizer → Hyperband HPO → pretraining → LoRA fine-tuning → FAISS index construction

In [ ]:
# Install dependencies into the active kernel's Python environment
import sys
# Remove old torch from user local packages directly
!rm -rf /home/jgu2930/.local/lib/python3.13/site-packages/torch*
# Install torch compiled for CUDA 12.4 (compatible with CUDA 12.8 driver)
!{sys.executable} -m pip install --index-url https://download.pytorch.org/whl/cu124 torch==2.5.1
!{sys.executable} -m pip install numpy datasets tokenizers sentence-transformers ray[tune] kaggle faiss-gpu-cu12 ipywidgets

In [ ]:
# Install transformers for GPT-2 fallback pipeline
import sys
!{sys.executable} -m pip install transformers

In [ ]:
# Redirect HuggingFace cache to project directory (more storage than home)
import os
os.environ["HF_HOME"] = "/projects/e32706/jgu2930/.cache/huggingface"
os.environ["HF_DATASETS_CACHE"] = "/projects/e32706/jgu2930/.cache/huggingface/datasets"

# Kaggle credentials via environment variable (bypasses file permission issues in Singularity)
os.environ["KAGGLE_USERNAME"] = "spencerbuehlman864"
os.environ["KAGGLE_KEY"] = ""

# Auto-reload loreforge.py on changes — no kernel restart needed
%load_ext autoreload
%autoreload 2

# Imports — all pipeline functions live in loreforge.py
from loreforge import run_training_pipeline
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Imports — all pipeline functions live in loreforge.py
from loreforge import run_training_pipeline
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Configure and run the full training pipeline
# Adjust parameters as needed based on Quest GPU allocation

UNIVERSES = ["star_wars", "harry_potter", "lotr"]

trained_model = run_training_pipeline(
    universes=UNIVERSES,
    n_hyperband_samples=20,   # number of Hyperband trials
    pretrain_max_epochs=10,   # max epochs per Hyperband trial + full pretraining
    finetune_epochs=3,        # LoRA fine-tuning epochs per universe
    finetune_lr=1e-4,         # LoRA fine-tuning learning rate
)

print("Training pipeline complete.")

In [ ]:

# =============================================================================
# INFERENCE VERIFICATION
# Load trained model + adapters and run a test generation for each universe
# =============================================================================

import torch
import json
import pathlib
from loreforge import (
    LoreForgeTransformer, load_tokenizer, apply_lora_adapters,
    load_lora_adapter, load_faiss_index, generate_story,
    ROOT_DIR, CHECKPOINTS_DIR, TOKENIZER_PATH
)

# Load best config
with open(ROOT_DIR / "best_config.json") as f:
    best_config = json.load(f)

PRETRAIN_EPOCHS = 2  # match pretrain_max_epochs used in training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = load_tokenizer(TOKENIZER_PATH)

TEST_PROMPTS = {
    "star_wars":    "Luke stared out at the twin suns of Tatooine and wondered about Obi Wan and his father",
    "harry_potter": "Harry and Hagrid get in a bar fight with dobby and Draco Malfoy",
    "lotr":         "Frodo looked upon the darkness of Mordor and felt love for Sam",
}

UNIVERSES = ["star_wars", "harry_potter", "lotr"]

for universe in UNIVERSES:
    print(f"\n{'='*60}")
    print(f"Universe: {universe.upper()}")
    print(f"{'='*60}")

    # Load base model with LoRA adapter
    model = LoreForgeTransformer(
        vocab_size=best_config["vocab_size"],
        d_model=best_config["d_model"],
        n_layers=best_config["n_layers"],
        n_heads=best_config["n_heads"],
        context_len=best_config["context_len"],
        dropout=0.0,
    )
    state_dict = torch.load(
        CHECKPOINTS_DIR / f"pretrain_epoch{PRETRAIN_EPOCHS}.pt",
        weights_only=True,
    )
    model.load_state_dict(state_dict)
    model = apply_lora_adapters(model)
    model = load_lora_adapter(model, universe, CHECKPOINTS_DIR)
    model = model.to(device)
    model.eval()

    # Load FAISS index
    faiss_index, passages = load_faiss_index(universe)

    # Generate
    prompt = TEST_PROMPTS[universe]
    print(f"Prompt: {prompt}\n")
    result = generate_story(
        prompt=prompt,
        universe=universe,
        model=model,
        tokenizer=tokenizer,
        faiss_index=faiss_index,
        passages=passages,
        max_new_tokens=200,
        temperature=0.9,
        top_k=50,
        device=device,
    )

    print(f"Generated:\n{result['generated_text']}")
    print(f"\nRetrieved passages ({len(result['retrieved_passages'])}):")
    for i, p in enumerate(result['retrieved_passages']):
        print(f"  [{i+1}] {p[:100]}...")


## GPT-2 Fallback Pipeline
Run the cells below if pivoting to the pretrained GPT-2 base model instead of the scratch-trained model.

In [ ]:
# GPT-2 Fine-tuning + RAG Pipeline
from loreforge_gpt2 import run_gpt2_pipeline

UNIVERSES = ["star_wars", "harry_potter", "lotr"]

model, tokenizer = run_gpt2_pipeline(
    universes=UNIVERSES,
    model_name="gpt2",       # options: "gpt2", "gpt2-medium", "gpt2-large"
    finetune_epochs=3,
    finetune_lr=1e-4,
)

print("GPT-2 pipeline complete.")